In [ ]:
import subprocess
subprocess.run(
    ['pip', 'install', 'huggingface_hub', 'sentencepiece', 'pytest', 'zstandard'],
    capture_output=False
)

In [ ]:
subprocess.run(
    ['/venv/main/bin/python', '-c',
     'import torch; print(torch.__version__, torch.cuda.get_device_name(0)); '
     'print("GPUs:", torch.cuda.device_count())'],
    capture_output=False
)

In [ ]:
from huggingface_hub import login, hf_hub_download
import os

# Replace with your HuggingFace token
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
login(token=HF_TOKEN)

os.makedirs('data/tokenizers', exist_ok=True)
os.makedirs('data/datasets/fineweb10B_sp1024', exist_ok=True)

hf_hub_download(
    repo_id='sproos/parameter-golf-tokenizers',
    filename='tokenizers/fineweb_1024_bpe.model',
    repo_type='model',
    local_dir='data'
)
hf_hub_download(
    repo_id='sproos/parameter-golf-tokenizers',
    filename='datasets/fineweb10B_sp1024/fineweb_val_000000.bin',
    repo_type='model',
    local_dir='data'
)
hf_hub_download(
    repo_id='sproos/parameter-golf-tokenizers',
    filename='datasets/fineweb10B_sp1024/fineweb_train_000000.bin',
    repo_type='model',
    local_dir='data'
)

In [ ]:
import os

REPO_DIR = 'modded-nanogpt'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=False)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/CameronBadman/modded-nanogpt.git'],
        capture_output=False
    )

# Show current HEAD
subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-5'], capture_output=False)

In [ ]:
# Install into the venv that has torch, then run smoke tests
subprocess.run(
    ['/venv/main/bin/pip', 'install', 'pytest', 'sentencepiece', 'zstandard'],
    capture_output=False
)
subprocess.run(
    ['/venv/main/bin/python', '-m', 'pytest', f'{REPO_DIR}/smoke_test.py', '-v'],
    capture_output=False
)

In [ ]:
import shutil, subprocess, os
shutil.rmtree('/tmp/torchinductor_root', ignore_errors=True)

env = os.environ.copy()

# --- Data paths ---
env['DATA_PATH']       = '/data/datasets/fineweb10B_sp1024'
env['TOKENIZER_PATH']  = '/data/tokenizers/fineweb_1024_bpe.model'

# --- Competition format: 10 min / 8xH100 ---
env['MAX_WALLCLOCK_SECONDS'] = '600'
env['ITERATIONS']            = '2500'
env['WARMDOWN_ITERS']        = '400'
env['WARMUP_STEPS']          = '50'
env['TRAIN_BATCH_TOKENS']    = '524288'

# --- Architecture ---
env['TRAIN_SEQ_LEN']         = '2048'   # longer context
env['MLP_MULT']              = '3'      # 3x MLP (freed by int6)
env['MODEL_DIM']             = '1536'
env['NUM_HEADS']             = '12'
env['NUM_KV_HEADS']          = '4'
env['NUM_RECURRENT_PASSES']  = '24'
env['DELTA_RANK']            = '8'

# --- Quantization + eval ---
env['QUANT_BITS']            = '6'      # int6 + zstd
env['VAL_EVAL_STRIDE']       = '256'    # sliding-window final eval

# --- Optimizer ---
env['GRAD_CLIP_NORM']        = '1.0'
env['MUON_MOMENTUM']         = '0.95'

# --- Logging ---
env['VAL_LOSS_EVERY']        = '250'
env['TRAIN_LOG_EVERY']       = '50'

subprocess.run(
    ['/venv/main/bin/python', f'{REPO_DIR}/train_gpt.py'],
    env=env,
    capture_output=False
)

In [ ]:
# Print final val_bpb from the log
import glob

logs = sorted(glob.glob(f'{REPO_DIR}/logs/*.txt'), key=lambda p: os.path.getmtime(p))
if logs:
    latest = logs[-1]
    print(f'Log: {latest}')
    with open(latest) as f:
        for line in f:
            if any(k in line for k in ('val_bpb', 'final_quant', 'pre_quant', 'submission')):
                print(line, end='')